# Smart Meters in London: General Statistics / EDA

General Statistics of the Data section for the kick-off document.

- Number of records
- Number of features/columns, data types
- Missing values
- Sample distributions (histograms, pie charts)

In [9]:
import os
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

ModuleNotFoundError: No module named 'kagglehub'

## 1. Download the dataset

`kagglehub` caches this locally after the first download, so re-running this cell later won't re-download the full ~10GB.

In [ ]:
path = kagglehub.dataset_download("jeanmidev/smart-meters-in-london")
print("Path to dataset files:", path)

dataset_dir = Path(path)

## 2. Inspect directory structure

Before loading anything, see what's actually here — file names, counts, and sizes. Don't assume the file naming; confirm it from this output.

In [ ]:
all_files = sorted(dataset_dir.rglob("*"))

total_size_mb = 0
for item in all_files:
    if item.is_file():
        size_mb = item.stat().st_size / (1024 ** 2)
        total_size_mb += size_mb
        print(f"{item.relative_to(dataset_dir)}  —  {size_mb:,.1f} MB")

print(f"\nTotal size on disk: {total_size_mb / 1024:,.2f} GB")

## 3. Load the small metadata table(s) first

This dataset typically ships a household info file (tariff assignment, Acorn group) separately from the half-hourly readings. It's small — safe to load in full immediately.

**Adjust the filename below once Step 2 shows you the real name.**

In [ ]:
# TODO: replace with the actual filename you saw in Step 2's output
household_info_path = dataset_dir / "informations_households.csv"

household_info = pd.read_csv(household_info_path)
print("Shape:", household_info.shape)
household_info.head()

In [ ]:
household_info.dtypes

## 4. Load one half-hourly consumption block as a test

Load a single block file first to check structure and memory footprint before touching the rest. The readings are usually split across many `block_*.csv` files.

In [ ]:
block_files = sorted(dataset_dir.rglob("block_*.csv"))
print(f"Found {len(block_files)} block files")

sample_block = pd.read_csv(block_files[0])
print("Shape:", sample_block.shape)
sample_block.head()

In [ ]:
sample_block.dtypes

## 5. Memory footprint check

This tells you the real inflation factor for pandas on this specific data — use it to estimate whether loading all blocks at once is safe on your machine.

In [ ]:
on_disk_mb = block_files[0].stat().st_size / (1024 ** 2)
in_memory_mb = sample_block.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"On disk:   {on_disk_mb:,.1f} MB")
print(f"In memory: {in_memory_mb:,.1f} MB")
print(f"Inflation factor: {in_memory_mb / on_disk_mb:.2f}x")
print(f"\nEstimated full dataset in memory (x{len(block_files)} blocks): "
      f"{(in_memory_mb * len(block_files)) / 1024:,.1f} GB")

## 6. Missing values check

Check both standard NaNs and non-standard missing markers (this dataset is known to sometimes store missing readings as the literal string `"Null"`, which pandas won't catch automatically).

In [ ]:
print("Standard NaN counts:")
print(sample_block.isnull().sum())

print("\nUnique values in object-type columns (checking for non-standard missing markers):")
for col in sample_block.select_dtypes(include="object").columns:
    print(f"  {col}: {sample_block[col].unique()[:5]}")

## 7. Sample distribution — consumption histogram

Placeholder — adjust the column name once you know the real one from Step 4's dtypes output (often something like `energy(kWh/hh)`).

In [ ]:
# TODO: replace "energy(kWh/hh)" with the actual consumption column name
consumption_col = "energy(kWh/hh)"

consumption_numeric = pd.to_numeric(sample_block[consumption_col], errors="coerce")

plt.figure(figsize=(8, 5))
sns.histplot(consumption_numeric.dropna(), bins=50)
plt.title("Distribution of half-hourly consumption (sample block)")
plt.xlabel("kWh per half-hour")
plt.show()

## 8. Sample distribution — tariff type pie chart

Placeholder — adjust the column name once you've confirmed it from `household_info.dtypes` above (often something like `stdorToU`).

In [ ]:
# TODO: replace "stdorToU" with the actual tariff-type column name
tariff_col = "stdorToU"

tariff_counts = household_info[tariff_col].value_counts()

plt.figure(figsize=(5, 5))
plt.pie(tariff_counts, labels=tariff_counts.index, autopct="%1.1f%%")
plt.title("Household distribution by tariff type")
plt.show()

## Next steps

Once the placeholders above are confirmed against real column names:

1. Loop over all `block_*.csv` files to get the true total record count (don't hold them all in memory at once — accumulate a running count and running stats instead).
2. Apply the dtype optimizations found here (e.g. `float32`, `category`) when reading the full set.
3. Aggregate missing-value counts across all blocks, not just the sample.
4. Consider converting the full dataset to Parquet at this point — it's a legitimate preprocessing step to describe in the methodology section, and will make every later notebook faster.